<a href="https://colab.research.google.com/github/ssdeassis/CursoPilarPython/blob/main/Calculo_de_perfil_do_vento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fpdf pandas ipywidgets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 87.1 MB/s eta 0:00:00
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40761 sha256=27569ed113011830414da0f723c7ed19d365b524c0fd6978dc1453db6ce3ad67
  Stored in directory: /root/.cache/pip/wheels/aa/da/11/a3189f34ddc13c26a2d0f329eac46b728c7f31c39e4dc26243
Successfully built fpdf


In [2]:
import pandas as pd
from fpdf import FPDF
from IPython.display import display, HTML
import ipywidgets as widgets

# Matriz de Parâmetros da Tabela 2 - NBR 6123
TABELA_S2 = {
    1: {'A': (1.10, 0.060), 'B': (1.11, 0.065), 'C': (1.12, 0.070)},
    2: {'A': (1.00, 0.085), 'B': (1.00, 0.090), 'C': (1.00, 0.100)},
    3: {'A': (0.94, 0.100), 'B': (0.94, 0.105), 'C': (0.93, 0.115)},
    4: {'A': (0.86, 0.120), 'B': (0.84, 0.125), 'C': (0.82, 0.135)},
    5: {'A': (0.74, 0.150), 'B': (0.71, 0.160), 'C': (0.67, 0.175)},
}

def determinar_classe(maior_dimensao):
    if maior_dimensao <= 20.0: return 'A'
    elif maior_dimensao <= 50.0: return 'B'
    else: return 'C'

class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.set_text_color(31, 78, 121)
        self.cell(0, 10, 'Memória de Cálculo - Perfil de Vento NBR 6123', border=False, ln=True, align='C')
        self.line(10, 20, 200, 20)
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(128)
        self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'C')

def gerar_pdf(projeto, local, responsavel, v0, s1, s3, categoria, classe, b, p, resultados, intervalo):
    pdf = PDF()
    pdf.add_page()

    # Informações Gerais
    pdf.set_font('Arial', '', 11)
    pdf.set_text_color(50, 50, 50)
    pdf.cell(0, 6, f'Projeto: {projeto}', ln=True)
    pdf.cell(0, 6, f'Localização: {local}', ln=True)
    pdf.cell(0, 6, f'Responsável Técnico: {responsavel}', ln=True)
    pdf.ln(5)

    # Parâmetros Adotados
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 8, '1. Parâmetros Normativos Adotados', ln=True)
    pdf.set_font('Arial', '', 11)
    pdf.cell(0, 6, f'- Velocidade Básica (V0): {v0} m/s', ln=True)
    pdf.cell(0, 6, f'- Categoria do Terreno: {categoria}', ln=True)
    pdf.cell(0, 6, f'- Classe da Edificação: {classe}', ln=True)
    pdf.cell(0, 6, f'- Fatores Calculados: b = {b} | p = {p}', ln=True)
    pdf.ln(5)

    # Tabela
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 8, f'2. Perfil de Velocidade (Intervalo de {intervalo} m)', ln=True)
    pdf.ln(2)

    pdf.set_font('Arial', 'B', 10)
    pdf.set_fill_color(31, 78, 121)
    pdf.set_text_color(255, 255, 255)
    pdf.cell(30, 8, 'Altura Z (m)', 1, 0, 'C', fill=True)
    pdf.cell(90, 8, 'Descrição', 1, 0, 'C', fill=True)
    pdf.cell(30, 8, 'Fator S2', 1, 0, 'C', fill=True)
    pdf.cell(40, 8, 'Vel. Vk (m/s)', 1, 1, 'C', fill=True)

    pdf.set_font('Arial', '', 10)
    pdf.set_text_color(0, 0, 0)
    fill = False

    for r in resultados:
        pdf.set_fill_color(242, 246, 249) if fill else pdf.set_fill_color(255, 255, 255)
        pdf.cell(30, 8, f"{r['Altura Z (m)']:.2f}", 1, 0, 'C', fill=True)
        pdf.cell(90, 8, r['Descrição'], 1, 0, 'L', fill=True)
        pdf.cell(30, 8, f"{r['Fator S2']:.3f}", 1, 0, 'C', fill=True)
        pdf.set_font('Arial', 'B', 10)
        pdf.cell(40, 8, f"{r['Velocidade Vk (m/s)']:.2f}", 1, 1, 'C', fill=True)
        pdf.set_font('Arial', '', 10)
        fill = not fill

    nome_arquivo = f"Vento_{projeto.replace(' ', '_')}.pdf"
    pdf.output(nome_arquivo)
    return nome_arquivo

In [3]:
def processar_interface(Projeto, Local, Responsavel, V0, S1, S3, Categoria, Altura_Total, Maior_Dimensao_Planta, Intervalo_Geracao):
    # Definir Classe e Parâmetros
    maior_dimensao_global = max(Altura_Total, Maior_Dimensao_Planta)
    classe_estrutura = determinar_classe(maior_dimensao_global)
    b, p = TABELA_S2[Categoria][classe_estrutura]

    # Gerar Cotas (Z)
    lista_z = []
    z_atual = 0.0
    while z_atual < Altura_Total:
        lista_z.append(z_atual)
        z_atual += Intervalo_Geracao

    lista_z.extend([5.0, Altura_Total])
    lista_z = sorted(list(set(lista_z))) # Remove duplicatas e ordena

    # Calcular Vk
    resultados = []
    for z in lista_z:
        if z == 0: desc = "Nível do terreno"
        elif z == Altura_Total: desc = "Topo da estrutura"
        elif z == 5.0 and 5.0 not in [i * Intervalo_Geracao for i in range(100)]: desc = "Cota mínima (Norma)"
        else: desc = f"Cota intermediária ({z:.2f} m)"

        z_efetivo = max(z, 5.0) if Categoria > 1 else z
        s2 = b * (z_efetivo / 10.0) ** p
        vk = V0 * S1 * s2 * S3

        resultados.append({
            "Altura Z (m)": z,
            "Descrição": desc,
            "Fator S2": round(s2, 3),
            "Velocidade Vk (m/s)": round(vk, 2)
        })

    # Mostrar Resultados no Notebook com Pandas
    df = pd.DataFrame(resultados)

    html_out = f"<h3>Resultados: Classe {classe_estrutura} (b={b}, p={p})</h3>"
    display(HTML(html_out))
    display(df)

    # Gerar PDF
    nome_pdf = gerar_pdf(Projeto, Local, Responsavel, V0, S1, S3, Categoria, classe_estrutura, b, p, resultados, Intervalo_Geracao)
    display(HTML(f"<br><b style='color:green;'>✅ Relatório salvo com sucesso: {nome_pdf}</b>"))

# Criando a interface interativa (Widgets)
interface = widgets.interact_manual(
    processar_interface,
    Projeto="Reservatório Elevado",
    Local="Codó - MA",
    Responsavel="Eng. Sergio Santos de Assis",
    V0=widgets.FloatSlider(min=20, max=50, step=1, value=30, description='V0 (m/s):'),
    S1=widgets.Dropdown(options=[1.0, 1.1], value=1.0, description='Fator S1:'),
    S3=widgets.Dropdown(options=[0.95, 1.0, 1.1], value=1.0, description='Fator S3:'),
    Categoria=widgets.Dropdown(options=[1, 2, 3, 4, 5], value=4, description='Categoria:'),
    Altura_Total=widgets.FloatText(value=25.22, description='Altura (m):'),
    Maior_Dimensao_Planta=widgets.FloatText(value=12.0, description='Base (m):'),
    Intervalo_Geracao=widgets.FloatSlider(min=1, max=10, step=0.5, value=3.0, description='Malha (m):')
)
interface.widget.children[-1].description = 'Calcular e Gerar PDF'
interface.widget.children[-1].button_style = 'primary'

interactive(children=(Text(value='Reservatório Elevado', description='Projeto'), Text(value='Codó - MA', descr…